In [1]:
import pandas as pd

In [2]:
sales = pd.DataFrame({
    "sale_id": [101,102,103,104,105,106,107,108,109,110,111,112],
    "sale_date": [
        "2026-01-05","2026-01-10","2026-01-15",
        "2026-02-03","2026-02-11","2026-02-20",
        "2026-01-07","2026-01-18","2026-02-06",
        "2026-02-14","2026-03-02","2026-03-05"
    ],
    "region": [
        "East","East","West","East","West","West",
        "South","South","South","East","West","South"
    ],
    "category": [
        "Laptop","Phone","Laptop","Laptop","Phone","Laptop",
        "Phone","Laptop","Laptop","Phone","Phone","Phone"
    ],
    "quantity": [2,3,1,1,4,2,2,1,3,2,5,4],
    "unit_price": [
        50000,20000,52000,51000,21000,50000,
        19000,49000,50000,20000,22000,19500
    ],
    "status": [
        "Completed","Completed","Completed","Completed",
        "Cancelled","Completed","Completed","Completed",
        "Completed","Completed","Completed","Cancelled"
    ]
})

In [3]:
sales.head()

,sale_id,sale_date,region,category,quantity,unit_price,status
0,101,2026-01-05,East,Laptop,2,50000,Completed
1,102,2026-01-10,East,Phone,3,20000,Completed
2,103,2026-01-15,West,Laptop,1,52000,Completed
3,104,2026-02-03,East,Laptop,1,51000,Completed
4,105,2026-02-11,West,Phone,4,21000,Cancelled


In [4]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   sale_id     12 non-null     int64
 1   sale_date   12 non-null     str  
 2   region      12 non-null     str  
 3   category    12 non-null     str  
 4   quantity    12 non-null     int64
 5   unit_price  12 non-null     int64
 6   status      12 non-null     str  
dtypes: int64(3), str(4)
memory usage: 804.0 bytes


In [5]:
sales[sales.duplicated()]

,sale_id,sale_date,region,category,quantity,unit_price,status


Converts sale_date to datetime.

In [7]:
sales['sale_date'] = pd.to_datetime(sales['sale_date'])

In [9]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   sale_id     12 non-null     int64         
 1   sale_date   12 non-null     datetime64[us]
 2   region      12 non-null     str           
 3   category    12 non-null     str           
 4   quantity    12 non-null     int64         
 5   unit_price  12 non-null     int64         
 6   status      12 non-null     str           
dtypes: datetime64[us](1), int64(3), str(3)
memory usage: 804.0 bytes


In [14]:
completed_sales = sales[sales['status'] == 'Completed']

In [15]:
completed_sales.head()

,sale_id,sale_date,region,category,quantity,unit_price,status
0,101,2026-01-05,East,Laptop,2,50000,Completed
1,102,2026-01-10,East,Phone,3,20000,Completed
2,103,2026-01-15,West,Laptop,1,52000,Completed
3,104,2026-02-03,East,Laptop,1,51000,Completed
5,106,2026-02-20,West,Laptop,2,50000,Completed


Creates revenue = quantity * unit_price.

In [16]:
completed_sales['revenue'] = completed_sales['quantity'] * completed_sales['unit_price']

In [17]:
completed_sales.head()

,sale_id,sale_date,region,category,quantity,unit_price,status,revenue
0,101,2026-01-05,East,Laptop,2,50000,Completed,100000
1,102,2026-01-10,East,Phone,3,20000,Completed,60000
2,103,2026-01-15,West,Laptop,1,52000,Completed,52000
3,104,2026-02-03,East,Laptop,1,51000,Completed,51000
5,106,2026-02-20,West,Laptop,2,50000,Completed,100000


Creates a month column from sale_date.

In [20]:
completed_sales['month'] = completed_sales['sale_date'].dt.month_name()

In [21]:
completed_sales.head()

,sale_id,sale_date,region,category,quantity,unit_price,status,revenue,month
0,101,2026-01-05,East,Laptop,2,50000,Completed,100000,January
1,102,2026-01-10,East,Phone,3,20000,Completed,60000,January
2,103,2026-01-15,West,Laptop,1,52000,Completed,52000,January
3,104,2026-02-03,East,Laptop,1,51000,Completed,51000,February
5,106,2026-02-20,West,Laptop,2,50000,Completed,100000,February


Produces a regional monthly revenue report:

- rows → region
- columns → month
- values → total revenue
- missing region/month combinations → 0

In [23]:
regional_monthly_report = completed_sales.pivot_table(
    index='region',
    columns='month',
    values='revenue',
    aggfunc='sum',
    fill_value=0
)

In [24]:
regional_monthly_report

month,February,January,March
region,,,
East,91000,160000,0
South,150000,87000,0
West,100000,52000,110000


Produces a second report:

- rows → region
- columns → category
- values → total revenue

In [26]:
regional_category_report = completed_sales.pivot_table(
    index='region',
    columns='category',
    values='revenue',
    aggfunc='sum'
)

In [28]:
regional_category_report

category,Laptop,Phone
region,,
East,151000,100000
South,199000,38000
West,152000,110000


Convert the regional monthly report from #5 back into long format with columns:
region | month | revenue

In [34]:
final_report = regional_monthly_report.reset_index().melt(
    id_vars='region',
    var_name='month',
    value_name='revenue'
)

In [35]:
final_report

,region,month,revenue
0,East,February,91000
1,South,February,150000
2,West,February,100000
3,East,January,160000
4,South,January,87000
5,West,January,52000
6,East,March,0
7,South,March,0
8,West,March,110000


Validation

In [36]:
completed_sales['revenue'].sum()

np.int64(750000)

In [37]:
final_report['revenue'].sum()

np.int64(750000)